# 04 - Piloto de escalamiento multi-anio (2019-2024)
Prueba de escalamiento del pipeline antes de extender al historico completo 2006-2023.

NOTA: 2024 son cifras PRELIMINARES segun INEGI (pueden ajustarse al publicarse la version definitiva).
Documentar esto en quality_report.md al momento de reportar resultados de ese anio.

IMPORTANTE - reorganiza data/raw/ por anio antes de correr este notebook:
```
data/raw/2019/DEFUN19.dbf, CATEMLDE19.dbf (o el nombre que traiga ese anio), ...
data/raw/2020/DEFUN20.dbf, ...
data/raw/2021/DEFUN21.dbf, ...
data/raw/2022/DEFUN22.dbf, ...
data/raw/2023/DEFUN23.dbf, ... (ya los tienes, solo mueve la carpeta)
data/raw/2024/DEFUN24.dbf, ... (cifras preliminares)
```


In [ ]:
import pandas as pd
import sys
sys.path.append('../src')
from cleaning_utils import (
    load_dbf, normalize_columns, apply_column_aliases, audit_year_schema, audit_multiple_years,
    load_and_filter_year, consolidate_years, recode_null_codes, split_edad, null_summary
)

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]


## 1. Auditoria de esquema (CRITICO - correr antes de consolidar)
Revisa si los 5 anios tienen las mismas columnas que el diccionario documentado.
Columnas 'faltantes' o 'extra' NO son necesariamente un error, pero deben documentarse
antes de continuar (ej. Afromex/Conindig se basan en el Censo 2020, es posible que
no existan en anios anteriores a esa actualizacion).

In [ ]:
resumen_esquema = audit_multiple_years(YEARS)
resumen_esquema


In [ ]:
# Revisar en detalle que columnas faltan/sobran por anio
for _, fila in resumen_esquema.iterrows():
    if fila['faltantes'] or fila['extra_no_documentadas']:
        print(f"Anio {fila['anio']}:")
        print(f"  Faltantes: {fila['faltantes']}")
        print(f"  Extra no documentadas: {fila['extra_no_documentadas']}")
        print()


## DECISION REQUERIDA ANTES DE CONTINUAR
Si la celda anterior mostro columnas faltantes o extra, DETENTE aqui y decide:
- Si son columnas menores (ej. una variable nueva agregada en anios recientes): documentar en
  data_dictionary.md y continuar, esas columnas quedaran como NaN en los anios donde no existan.
- Si son diferencias estructurales grandes: puede requerir logica de mapeo especial por anio.

Si el esquema es consistente (sin diferencias), continuar directo a la seccion 2.

## 2. Consolidar los 5 anios (carga + filtro de suicidio)

In [ ]:
df_multi = consolidate_years(YEARS)


## 3. Validacion rapida: tendencia por anio
Antes de limpiar, verificar que el volumen por anio tenga sentido (sin caidas o picos extranos
que sugieran un problema de carga, mas alla de la variacion real esperada).

In [ ]:
casos_por_anio = df_multi['anio_dataset'].value_counts().sort_index()
casos_por_anio


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
casos_por_anio.plot(kind='bar', color='#4C72B0', ax=ax)
ax.set_title('Casos de suicidio por anio (piloto 2019-2024)')
ax.set_xlabel('')
ax.set_ylabel('Numero de casos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../docs/figuras/09_casos_por_anio_piloto.png')
plt.show()


## 4. Aplicar reglas de limpieza (reutilizando funciones ya validadas en 02_cleaning)

In [ ]:
df_multi_limpio = recode_null_codes(df_multi)
df_multi_limpio = split_edad(df_multi_limpio, col='Edad')


## 5. Guardar dataset consolidado

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_multi_limpio.to_csv('../data/processed/suicidio_2019_2024_consolidado.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df_multi_limpio):,} registros, anios {YEARS}')


In [1]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/suicidio_2019_2024_consolidado.csv',
    encoding='utf-8', low_memory=False, dtype=str
)

# Confirmar nombre de columna de anio
[c for c in df.columns if 'anio' in c.lower() or 'año' in c.lower() or 'year' in c.lower()]

['Anio_ocur', 'Anio_regis', 'Anio_nacim', 'Anio_cert', 'anio_dataset']

In [2]:
tabla_por_anio = (
    df['anio_dataset']
    .value_counts()
    .sort_index()
    .rename('n_registros')
    .reset_index()
    .rename(columns={'index': 'anio'})
)

print(tabla_por_anio)
print(f'\nTotal 2019-2024: {tabla_por_anio["n_registros"].sum():,} registros')

  anio_dataset  n_registros
0         2019         7225
1         2020         7896
2         2021         8433
3         2022         8241
4         2023         9072
5         2024         9051

Total 2019-2024: 49,918 registros


## 5b. Validacion de codigos geograficos por anio
Repite la validacion hecha en 03_validation.ipynb (solo para 2023) sobre cada
uno de los 6 anios del consolidado, para confirmar que no hay codigos de
entidad huerfanos en ningun anio individual.

In [ ]:
from cleaning_utils import CATALOG_CANONICAL_COLUMNS, validate_codes_against_catalog

resultados_geo_por_anio = []
for anio in sorted(df_multi_limpio['anio_dataset'].unique()):
    yy = str(anio)[-2:]
    cat_geo_anio = load_dbf(f'../data/raw/{anio}/CATEMLDE{yy}.dbf')
    cat_geo_anio = normalize_columns(cat_geo_anio, canonical_names=CATALOG_CANONICAL_COLUMNS)
    cat_entidades_anio = cat_geo_anio[(cat_geo_anio['Cve_mun'] == '000') & (cat_geo_anio['Cve_loc'] == '0000')]

    sub = df_multi_limpio[df_multi_limpio['anio_dataset'] == anio]
    resultado = validate_codes_against_catalog(sub, 'Ent_ocurr', cat_entidades_anio, 'Cve_ent')
    resultados_geo_por_anio.append({
        'anio': anio,
        'codigos_huerfanos': resultado['n_codigos_huerfanos'],
        'registros_afectados': resultado['n_registros_afectados'],
    })

df_geo_por_anio = pd.DataFrame(resultados_geo_por_anio).set_index('anio')
df_geo_por_anio


## 6. Hallazgos del piloto de escalamiento
_Documentar aqui: diferencias de esquema encontradas, si la tendencia por anio tiene sentido,
y que ajustes se necesitarian para escalar al historico completo 2006-2023 (ej. anios muy
anteriores pueden requerir mapeo de columnas distinto, o el nombre de archivo puede variar)._